In [4]:
from typing import TypedDict, Annotated 
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
import operator

MAX_TRIES = 3
PASS_SCORE = 70


class State(TypedDict):
    ticket: str
    category: str
    risk: str
    draft: str
    score: int
    tries: int
    status: str
    log: Annotated[list[str], operator.add]

In [5]:
def classify(state: State) -> dict:
    t = state["ticket"].lower()
    category = "billing" if ("charge" in t or "refund" in t) else "bug" if "error" in t else "other"
    risk = "high" if  ("refund" in t or "legal" in t) else "low"
    return {"category": category, "risk": risk, "log": [f"classify -> {category}/{risk}"]}

In [6]:
def draft(state: State) -> dict:
    n = state["tries"] + 1
    return {"draft": f"reply v{n}", "tries": n, "log": [f"draft v{n}"]}

In [19]:
def judge(state: State) -> dict:
    score = 0 if "unfixable" in state["ticket"].lower() else 40*state["tries"]
    return {"score": score, "log": [f"judge score {score}"]}

In [20]:
def human_review(state: State) -> dict:
    answer = interrupt({"ticket": state["ticket"], "draft": state["draft"]})
    ok = answer == "approve"
    return {
        "status": "sent after human approval" if ok else "rejected by human",
        "log": [f"human said {answer}"]
    }

In [21]:
def finalize(state: State) -> dict:
    return {"status": "auto-agent", "log":["finalize"]}

In [22]:
def escalate(state: State) -> dict:
    return {"status": "esclated to a human (draft kept failing)", "log": ["escalate"]}



In [23]:
from typing import Literal


def after_judge(state: State) -> Literal["draft", "human_review", "finalize", "escalate"]:
    if state["score"] >= PASS_SCORE:
        return "human_review" if state["risk"] == "high" else "finalize"
    return "escalate" if state["tries"] >= MAX_TRIES else "draft"
        

In [24]:
def build(checkpointer=None):
    g = StateGraph(State)
    for name, fn in [("classify", classify), ("draft", draft), ("judge", judge),
                     ("human_review", human_review), ("finalize", finalize), ("escalate", escalate)]:
        g.add_node(name, fn)
    g.add_edge(START, "classify")
    g.add_edge("classify", "draft")
    g.add_edge("draft", "judge")
    g.add_conditional_edges("judge", after_judge)
    g.add_edge("human_review", END)
    g.add_edge("finalize", END)
    g.add_edge("escalate", END)
    return g.compile(checkpointer=checkpointer)



In [25]:
def fresh(ticket):
    return {"ticket": ticket, "category": "", "risk": "", "draft": "", "score": 0, "tries": 0, "status": "", "log": []}

In [26]:
# unit tests for the router: no graph, no model 

base = fresh("x")
assert after_judge({**base, "score":80, "risk": "low"}) == "finalize"
assert after_judge({**base, "score":80, "risk": "high"}) == "human_review"
assert after_judge({**base, "score":10, "tries": 1}) == "draft"
assert after_judge({**base, "score":10, "tries": 3}) == "escalate"

print(" router tested: 4 passed \n")

 router tested: 4 passed 



In [27]:
graph = build(InMemorySaver())

In [28]:
print("A) low-risk bug ticket")
r = graph.invoke(fresh("App shows an error on login"), {"configurable": {"thread_id": "A"}})
print("   status:", r["status"]); print("   log   :", r["log"], "\n")

A) low-risk bug ticket
   status: auto-agent
   log   : ['classify -> bug/low', 'draft v1', 'judge score 40', 'draft v2', 'judge score 80', 'finalize'] 



In [29]:
print("B) high-risk refund ticket (pauses for a human)")
cfg = {"configurable": {"thread_id": "B"}}
r = graph.invoke(fresh("Please refund my duplicate charge"), cfg)
print("   paused, next node:", graph.get_state(cfg).next)
r = graph.invoke(Command(resume="approve"), cfg)
print("   status:", r["status"]); print("   log   :", r["log"], "\n")

B) high-risk refund ticket (pauses for a human)
   paused, next node: ('human_review',)
   status: sent after human approval
   log   : ['classify -> billing/high', 'draft v1', 'judge score 40', 'draft v2', 'judge score 80', 'human said approve'] 



In [30]:
print("C) unfixable ticket (judge always fails)")
r = graph.invoke(fresh("This is unfixable, legal threat"), {"configurable": {"thread_id": "C"}})
print("   status:", r["status"]); print("   log   :", r["log"])
print()

C) unfixable ticket (judge always fails)
   status: esclated to a human (draft kept failing)
   log   : ['classify -> other/high', 'draft v1', 'judge score 0', 'draft v2', 'judge score 0', 'draft v3', 'judge score 0', 'escalate']

